In [1]:
import pandas as pd

dfs = []
for year in range(2014, 2019):
    d = pd.read_csv(f"../Data/{year}_Financial_Data.csv")
    d = d.rename(columns={"Unnamed: 0": "Ticker", f"{year+1} PRICE VAR [%]": "PRICE VAR [%]"})
    d["Year"] = year
    d["Target"] = (d["PRICE VAR [%]"] > d["PRICE VAR [%]"].median()).astype(int)
    dfs.append(d)

C:\Users\User\AppData\Local\Temp\ipykernel_33400\3823477683.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["Year"] = year
C:\Users\User\AppData\Local\Temp\ipykernel_33400\3823477683.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["Target"] = (d["PRICE VAR [%]"] > d["PRICE VAR [%]"].median()).astype(int)
C:\Users\User\AppData\Local\Temp\ipykernel_33400\3823477683.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Con

In [2]:
all_data = pd.concat(dfs, ignore_index=True)
all_data.shape

(22077, 227)

In [3]:
train = all_data[all_data["Year"] <= 2016]
val = all_data[all_data["Year"] == 2017]
test = all_data[all_data["Year"] == 2018]

In [4]:
all_data["Year"].value_counts().sort_index()

Year
2014    3808
2015    4120
2016    4797
2017    4960
2018    4392
Name: count, dtype: int64

In [5]:
print(train.shape, val.shape, test.shape)

(12725, 227) (4960, 227) (4392, 227)


Data uppdelad efter tid: träning 2014-2016 (12 725 rader), validering 2017 (4 960), test 2018 (4 392). Target skapad per år med årets median.

In [6]:
for kolumn in all_data.columns:
    print(kolumn)

Ticker
Revenue
Revenue Growth
Cost of Revenue
Gross Profit
R&D Expenses
SG&A Expense
Operating Expenses
Operating Income
Interest Expense
Earnings before Tax
Income Tax Expense
Net Income - Non-Controlling int
Net Income - Discontinued ops
Net Income
Preferred Dividends
Net Income Com
EPS
EPS Diluted
Weighted Average Shs Out
Weighted Average Shs Out (Dil)
Dividend per Share
Gross Margin
EBITDA Margin
EBIT Margin
Profit Margin
Free Cash Flow margin
EBITDA
EBIT
Consolidated Income
Earnings Before Tax Margin
Net Profit Margin
Cash and cash equivalents
Short-term investments
Cash and short-term investments
Receivables
Inventories
Total current assets
Property, Plant & Equipment Net
Goodwill and Intangible Assets
Long-term investments
Tax assets
Total non-current assets
Total assets
Payables
Short-term debt
Total current liabilities
Long-term debt
Total debt
Deferred revenue
Tax Liabilities
Deposit Liabilities
Total non-current liabilities
Total liabilities
Other comprehensive income
Retain

In [7]:
bort = ["Target", "Class", "PRICE VAR [%]", "Ticker", "Year"]
kandidater = [k for k in all_data.columns if k not in bort]
len(kandidater)

222

In [8]:
saknas = train[kandidater].isna().mean().sort_values(ascending=False)
saknas.head(15)

operatingCycle                                0.999371
cashConversionCycle                           0.999371
shortTermCoverageRatios                       0.494931
10Y Shareholders Equity Growth (per Share)    0.442986
10Y Revenue Growth (per Share)                0.436699
10Y Net Income Growth (per Share)             0.436699
10Y Operating CF Growth (per Share)           0.433949
10Y Dividend per Share Growth (per Share)     0.418153
priceEarningsToGrowthRatio                    0.405265
dividendPayoutRatio                           0.405265
Net Debt to EBITDA                            0.403536
nIperEBT                                      0.384204
effectiveTaxRate                              0.384204
niperEBT                                      0.384204
Net Debt                                      0.365737
dtype: float64

In [9]:
nyckeltal = [
    "Net Profit Margin", "returnOnEquity", "returnOnAssets",
    "debtEquityRatio", "currentRatio", "interestCoverage",
    "priceEarningsRatio", "priceToBookRatio", "priceToSalesRatio",
    "Revenue Growth", "EPS Growth", "Asset Growth",
]
[k for k in nyckeltal if k not in all_data.columns]

[]

In [10]:
train[nyckeltal].isna().mean().sort_values(ascending=False)

returnOnAssets        0.293281
priceToBookRatio      0.216424
currentRatio          0.156778
Asset Growth          0.108527
EPS Growth            0.105147
returnOnEquity        0.097525
debtEquityRatio       0.096503
Revenue Growth        0.096267
priceToSalesRatio     0.093752
priceEarningsRatio    0.093595
interestCoverage      0.088644
Net Profit Margin     0.069548
dtype: float64

In [11]:
medianer = train[nyckeltal].median()
medianer

Net Profit Margin      0.044000
returnOnEquity         0.072250
returnOnAssets         0.045300
debtEquityRatio        0.419600
currentRatio           1.807076
interestCoverage       0.818600
priceEarningsRatio    13.779450
priceToBookRatio       1.764300
priceToSalesRatio      1.875350
Revenue Growth         0.043500
EPS Growth             0.075900
Asset Growth           0.045500
dtype: float64

In [12]:
X_train = train[nyckeltal].fillna(medianer)
X_val = val[nyckeltal].fillna(medianer)
X_test = test[nyckeltal].fillna(medianer)

In [13]:
y_train = train["Target"]
y_val = val["Target"]
y_test = test["Target"]

In [14]:
print(X_train.isna().sum().sum(), X_val.isna().sum().sum(), X_test.isna().sum().sum())

0 0 0


12 nyckeltal valda inom lönsamhet, skuldsättning, värdering och tillväxt. Saknade värden fyllda med medianen från träningsdatan (2014-2016), även i validering och test, för att undvika dataläckage.

In [15]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
accuracy_score(y_val, baseline.predict(X_val))

0.5

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [17]:
from sklearn.linear_model import LogisticRegression

modell = LogisticRegression(max_iter=1000)
modell.fit(X_train_s, y_train)
accuracy_score(y_val, modell.predict(X_val_s))

0.5405241935483871

Baseline (alltid gissa vanligaste klassen): 50 % accuracy på validering (2017). Logistisk regression med 12 nyckeltal (skalade med träningsdatans medelvärde och spridning): 54,1 % accuracy på validering.

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
accuracy_score(y_val, rf.predict(X_val))

0.5298387096774193

In [19]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
accuracy_score(y_val, gb.predict(X_val))

0.5243951612903226

In [20]:
resultat = {
    "Baseline": accuracy_score(y_val, baseline.predict(X_val)),
    "Logistisk regression": accuracy_score(y_val, modell.predict(X_val_s)),
    "Random forest": accuracy_score(y_val, rf.predict(X_val)),
    "Gradient boosting": accuracy_score(y_val, gb.predict(X_val)),
}
resultat

{'Baseline': 0.5,
 'Logistisk regression': 0.5405241935483871,
 'Random forest': 0.5298387096774193,
 'Gradient boosting': 0.5243951612903226}